## Weather

In [1]:
import pandas as pd

df = pd.read_csv('result.txt',
                 comment='#',
                 skipinitialspace=True,
                 header=0)

df.columns = ['STN', 'YYYYMMDD', 'TG', 'TN', 'TX', 'SQ', 'RH']

print(df.columns.tolist())
print(df.head(3))

['STN', 'YYYYMMDD', 'TG', 'TN', 'TX', 'SQ', 'RH']
   STN  YYYYMMDD  TG  TN  TX  SQ  RH
0  260  20170102  30  -3  68  43   5
1  260  20170103  50  15  71   5   7
2  260  20170104  58  26  73  33  18


In [2]:
df = df.rename(columns={'YYYYMMDD': 'date'})

df['date'] = pd.to_datetime(df['date'].astype(str), format='%Y%m%d')
df = df.sort_values('date').reset_index(drop=True)

df['TG'] = df['TG'] / 10
df['TN'] = df['TN'] / 10
df['TX'] = df['TX'] / 10
df['RH'] = df['RH'].clip(lower=0) / 10
df['SQ'] = df['SQ'].clip(lower=0) / 10

df = df.sort_values('date').reset_index(drop=True)
print(df.isnull().sum())
print(df.head(3))

STN     0
date    0
TG      0
TN      0
TX      0
SQ      0
RH      0
dtype: int64
   STN       date   TG   TN   TX   SQ   RH
0  260 2017-01-02  3.0 -0.3  6.8  4.3  0.5
1  260 2017-01-03  5.0  1.5  7.1  0.5  0.7
2  260 2017-01-04  5.8  2.6  7.3  3.3  1.8


In [3]:
df['year'] = df['date'].dt.isocalendar().year.astype(int)
df['week'] = df['date'].dt.isocalendar().week.astype(int)

df_weekly = df.groupby(['year', 'week']).agg(
    temp_mean=('TG', 'mean'),
    temp_min=('TN', 'min'),
    temp_max=('TX', 'max'),
    precip_sum=('RH', 'sum'),
    sunshine_sum=('SQ', 'sum')
).reset_index()

print(df_weekly.shape)
print(df_weekly.head(5))

(209, 7)
   year  week  temp_mean  temp_min  temp_max  precip_sum  sunshine_sum
0  2017     1   2.128571      -6.5       7.3         8.2          20.1
1  2017     2   4.014286      -1.8      10.4        38.1          14.0
2  2017     3  -1.857143      -7.4       5.1         0.0          36.0
3  2017     4   1.342857      -7.0       8.1         3.9          14.5
4  2017     5   5.614286      -1.1      12.5        17.1           9.7


In [4]:
week_avg = df_weekly.groupby('week')['temp_mean'].transform('mean')
df_weekly['temp_anomaly'] = (df_weekly['temp_mean'] - week_avg).round(2)
df_weekly['heavy_rain'] = (
    df_weekly['precip_sum'] > df_weekly['precip_sum'].quantile(0.9)
).astype(int)

print(df_weekly.head(5))

   year  week  temp_mean  temp_min  temp_max  precip_sum  sunshine_sum  \
0  2017     1   2.128571      -6.5       7.3         8.2          20.1   
1  2017     2   4.014286      -1.8      10.4        38.1          14.0   
2  2017     3  -1.857143      -7.4       5.1         0.0          36.0   
3  2017     4   1.342857      -7.0       8.1         3.9          14.5   
4  2017     5   5.614286      -1.1      12.5        17.1           9.7   

   temp_anomaly  heavy_rain  
0         -2.73           0  
1         -1.58           0  
2         -4.82           0  
3         -1.97           0  
4          0.66           0  


In [5]:
df_weekly.to_csv('weather_weekly.csv', index=False)
print("Saved: weather_weekly.csv")
print(df_weekly.shape)

Saved: weather_weekly.csv
(209, 9)


## Public Holiday

In [6]:
import holidays
import pandas as pd
import numpy as np

nl_holidays = holidays.Netherlands(years=range(2017, 2023))

df_holidays = pd.DataFrame([
    {'date': pd.Timestamp(d), 'holiday_name': name}
    for d, name in sorted(nl_holidays.items())
])

print(df_holidays)

         date             holiday_name
0  2017-01-01           New Year's Day
1  2017-04-14              Good Friday
2  2017-04-16            Easter Sunday
3  2017-04-17            Easter Monday
4  2017-04-27               King's Day
..        ...                      ...
56 2022-05-26            Ascension Day
57 2022-06-05              Whit Sunday
58 2022-06-06              Whit Monday
59 2022-12-25            Christmas Day
60 2022-12-26  Second Day of Christmas

[61 rows x 2 columns]


In [7]:
date_range = pd.date_range('2018-01-01', '2021-12-31', freq='D')
df_cal = pd.DataFrame({'date': date_range})

df_cal = df_cal.merge(df_holidays, on='date', how='left')
df_cal['is_holiday'] = df_cal['holiday_name'].notna().astype(int)

In [8]:
holiday_dates_list = df_holidays['date'].tolist()

df_cal['days_to_nearest'] = df_cal['date'].apply(
    lambda d: int(min(abs((d - h).days) for h in holiday_dates_list))
)

print(df_cal[['date', 'days_to_nearest']].iloc[95:115])

          date  days_to_nearest
95  2018-04-06                4
96  2018-04-07                5
97  2018-04-08                6
98  2018-04-09                7
99  2018-04-10                8
100 2018-04-11                9
101 2018-04-12               10
102 2018-04-13               11
103 2018-04-14               12
104 2018-04-15               12
105 2018-04-16               11
106 2018-04-17               10
107 2018-04-18                9
108 2018-04-19                8
109 2018-04-20                7
110 2018-04-21                6
111 2018-04-22                5
112 2018-04-23                4
113 2018-04-24                3
114 2018-04-25                2


In [9]:
holiday_map = {
    "New Year's Day"      : 'new_year',
    'Good Friday'         : 'easter',
    'Easter Sunday'       : 'easter',
    'Easter Monday'       : 'easter',
    "King's Day"          : 'kings_day',
    'Liberation Day'      : 'liberation_day',
    'Ascension Day'       : 'ascension',
    'Whit Sunday'         : 'pentecost',
    'Whit Monday'         : 'pentecost',
    'Christmas Day'       : 'christmas',
    'Second Christmas Day': 'christmas',
}

df_cal['holiday_type'] = df_cal['holiday_name'].map(holiday_map)
holiday_dummies = pd.get_dummies(df_cal['holiday_type'], prefix='hol').astype(int)
df_cal = pd.concat([df_cal, holiday_dummies], axis=1)

print(df_cal[df_cal['is_holiday'] == 1][['date', 'holiday_name', 'holiday_type']].head(10))

          date             holiday_name holiday_type
0   2018-01-01           New Year's Day     new_year
88  2018-03-30              Good Friday       easter
90  2018-04-01            Easter Sunday       easter
91  2018-04-02            Easter Monday       easter
116 2018-04-27               King's Day    kings_day
129 2018-05-10            Ascension Day    ascension
139 2018-05-20              Whit Sunday    pentecost
140 2018-05-21              Whit Monday    pentecost
358 2018-12-25            Christmas Day    christmas
359 2018-12-26  Second Day of Christmas          NaN


In [10]:
df_cal['year'] = df_cal['date'].dt.isocalendar().year.astype(int)
df_cal['week'] = df_cal['date'].dt.isocalendar().week.astype(int)

hol_cols = [c for c in df_cal.columns if c.startswith('hol_')]

df_holiday_weekly = df_cal.groupby(['year', 'week']).agg(
    has_holiday=('is_holiday', 'max'),
    min_days_to_holiday=('days_to_nearest', 'min')
).reset_index()

hol_agg = df_cal.groupby(['year', 'week'])[hol_cols].max().reset_index()
df_holiday_weekly = df_holiday_weekly.merge(hol_agg, on=['year', 'week'])

print(df_holiday_weekly.shape)
print(df_holiday_weekly.head(10))

(209, 11)
   year  week  has_holiday  min_days_to_holiday  hol_ascension  hol_christmas  \
0  2018     1            1                    0              0              0   
1  2018     2            0                    7              0              0   
2  2018     3            0                   14              0              0   
3  2018     4            0                   21              0              0   
4  2018     5            0                   28              0              0   
5  2018     6            0                   35              0              0   
6  2018     7            0                   40              0              0   
7  2018     8            0                   33              0              0   
8  2018     9            0                   26              0              0   
9  2018    10            0                   19              0              0   

   hol_easter  hol_kings_day  hol_liberation_day  hol_new_year  hol_pentecost  
0           0     

In [11]:
df_holiday_weekly.to_csv('holiday_weekly.csv', index=False)
print("Saved: holiday_weekly.csv")
print(df_holiday_weekly.columns.tolist())

Saved: holiday_weekly.csv
['year', 'week', 'has_holiday', 'min_days_to_holiday', 'hol_ascension', 'hol_christmas', 'hol_easter', 'hol_kings_day', 'hol_liberation_day', 'hol_new_year', 'hol_pentecost']
